# `edge_eeg` Colab GPU High-Speed Training (Teacher Probe & 8-Ch Student)
开机 5 秒从 Google Drive 解包特征缓存至本地 NVMe (`/content/local_cache`)，支持 306 通道 Teacher 探顶与 8 通道端侧 Student 训练，以及 `--resume auto` 断点续训。

In [ ]:
# === Cell 1: 挂载 Drive、拉取代码与建立 /content/studies_links 软链桥 ===
from google.colab import drive
import os
from pathlib import Path

drive.mount("/content/drive")
DRIVE_WS = Path("/content/drive/MyDrive/EdgeEEG_Workspace")
for sub in ("data", "archives", "runs"):
    (DRIVE_WS / sub).mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/<YOUR_USERNAME>/EmbeddedBrainDecoding.git"
CODE_DIR = Path("/content/code")
if not CODE_DIR.exists():
    !git clone {REPO_URL} {CODE_DIR}
else:
    !git -C {CODE_DIR} pull --ff-only

LINKS_DIR = Path("/content/studies_links")
LINKS_DIR.mkdir(parents=True, exist_ok=True)
for ds_name in ("libribrain", "auditory_eeg"):
    target = DRIVE_WS / "data" / ds_name
    target.mkdir(parents=True, exist_ok=True)
    link_path = LINKS_DIR / ds_name
    if link_path.is_symlink() or link_path.exists():
        link_path.unlink()
    os.symlink(target, link_path)
print("Symlink bridge ready:", list(LINKS_DIR.iterdir()))

In [ ]:
# === Cell 2: 初始化兼容 Python 3.13+ 的安全虚拟环境 ===
!python3 /content/code/scripts/colab_setup.py --venv /content/eeg-venv --repo /content/code
PYTHON = "/content/eeg-venv/bin/python"

In [ ]:
# === Cell 3: 秒级解包特征缓存至本地 NVMe (/content/local_cache) ===
import os
os.environ["EDGEEEG_STUDIES"] = "/content/studies_links"
os.environ["EDGEEEG_CACHE"] = "/content/local_cache"
os.environ["EDGEEEG_RESULTS"] = "/content/drive/MyDrive/EdgeEEG_Workspace/runs"

MODALITY = "meg"
ARCHIVE_PATH = f"/content/drive/MyDrive/EdgeEEG_Workspace/archives/features_{MODALITY}_sub001.tar"
!{PYTHON} -u /content/code/scripts/cache_archive.py unpack \
    --archive {ARCHIVE_PATH} \
    --cache-dir /content/local_cache

In [ ]:
# === Cell 4A: 第一阶探顶 — 306 通道 Teacher 大模型训练 ===
!{PYTHON} -u -m edge_eeg train \
    --modality meg \
    --stage phoneme \
    --arch teacher_conformer \
    --channels 306 \
    --subject 001 \
    --epochs 30 \
    --devices 1 \
    --require-cache \
    --resume auto

In [ ]:
# === Cell 4B: 第二阶端侧瘦身 — 颞叶 8 通道 Student 轻量模型训练 ===
!{PYTHON} -u -m edge_eeg train \
    --modality meg \
    --stage phoneme \
    --arch student_tiny \
    --channels 8 \
    --subject 001 \
    --epochs 50 \
    --devices 1 \
    --require-cache \
    --resume auto